# Tái hiện NVAUTO (bài 22, ISLES 2022) trên Kaggle

Auto3DSeg SegResNet, ADC + DWI, patch 192×192×128, Dice + Focal có deep supervision, AdamW 2e-4 cosine, 1000 epoch, 5-fold.

**Trước khi chạy:** Settings → Accelerator = *GPU T4 x2*, Internet = *On*; **Add Input** = dataset ISLES-2022 của nhóm. Code được tải từ GitHub (`UwUswain/PBL6`, nhánh `long_exp`), không cần tải lên dataset code.

Chạy bằng **Save Version → Save & Run All (Commit)** để chạy nền tới 12 giờ và giữ lại output.

Từ phiên thứ 2: Add Input → Your Work → output phiên trước, rồi đặt `RESUME_FROM` ở ô cấu hình.

In [ ]:
# ===== Cấu hình: chỉ cần sửa ô này =====
COMMIT = "3dfaf7f5f53d22f5b427c10005b52ec199ec0242"  # ghim code để mọi phiên chạy cùng một bản
FOLDS = "0"          # phiên đầu: chỉ fold 0 để đo tốc độ; sau đó "0 1 2 3 4"
EPOCHS = 1000        # bài 22: 1000
HOURS = 11.3         # tự dừng an toàn trước giới hạn 12 giờ của Kaggle
RESUME_FROM = ""     # từ phiên 2: vd "/kaggle/input/nvauto-run-1" (output phiên trước)

In [ ]:
# Lấy code từ GitHub và cài thư viện
!pip install -q "monai==1.6.1" nibabel fire tensorboard
!rm -rf /kaggle/working/PBL6 && git clone -q -b long_exp https://github.com/UwUswain/PBL6.git /kaggle/working/PBL6
!cd /kaggle/working/PBL6 && git checkout -q {COMMIT} && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Tìm dữ liệu; nếu dataset được lưu dạng nén (.sip/.zip) thì giải nén DWI, ADC và mask (bỏ FLAIR: không dùng, rất nặng)
import zipfile
from pathlib import Path
inp = Path("/kaggle/input")
root = next((d.parent for d in inp.rglob("derivatives") if d.is_dir() and any(d.parent.glob("sub-strokecase*"))), None)
if root is None:
    arc = sorted([*inp.rglob("*.sip"), *inp.rglob("*.zip")])
    assert arc, "Không tìm thấy ISLES-2022 trong /kaggle/input: hãy Add Input dataset của nhóm"
    with zipfile.ZipFile(arc[0]) as z:
        z.extractall("/tmp/isles", [m for m in z.namelist() if "/anat/" not in m])
    root = next(d.parent for d in Path("/tmp/isles").rglob("derivatives") if d.is_dir())
DATA_ROOT = str(root)
print("DATA_ROOT =", DATA_ROOT, "|", len(list(root.glob("sub-strokecase*"))), "ca")

In [ ]:
# Train: tự dừng trước HOURS giờ, checkpoint lưu mỗi epoch
resume = f"--resume-from {RESUME_FROM}" if RESUME_FROM else ""
!python /kaggle/working/PBL6/kaggle/nvauto_repro/run_nvauto.py train --folds {FOLDS} --epochs {EPOCHS} --hours {HOURS} --data-root {DATA_ROOT} {resume}

In [ ]:
# Chỉ chạy khi mọi fold đã "done": Dice từng ca ở độ phân giải gốc, trung bình + trung vị
# !python /kaggle/working/PBL6/kaggle/nvauto_repro/run_nvauto.py evaluate --folds {FOLDS} --epochs {EPOCHS} --data-root {DATA_ROOT} {resume}